In [1]:
!pip install -q monai nibabel einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 42.8 MB/s eta 0:00:00a 0:00:01


In [21]:
import os
import glob
import tarfile
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, CropForegroundd, RandCropByPosNegLabeld,
    RandFlipd, RandRotate90d, EnsureTyped, AsDiscrete, Lambdad
)

from monai.data import Dataset, DataLoader, decollate_batch
from monai.networks.nets import UNet
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

In [7]:
BASE_PATH = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
for root, dirs, files in os.walk(BASE_PATH):
    if len(files) > 0:
        print(root)
        print(files[:5])
        break

/kaggle/input/datasets/dschettler8845/brats-2021-task1
['BraTS2021_00495.tar', 'BraTS2021_Training_Data.tar', 'BraTS2021_00621.tar']


In [8]:
import tarfile
import os

BASE_PATH = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
EXTRACT_PATH = "/kaggle/working/brats2021"

os.makedirs(EXTRACT_PATH, exist_ok=True)

tar_path = os.path.join(BASE_PATH, "BraTS2021_Training_Data.tar")

with tarfile.open(tar_path, "r") as tar:
    tar.extractall(path=EXTRACT_PATH)

print("Extraction completed.")

/tmp/ipykernel_57/3219288561.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=EXTRACT_PATH)


Extraction completed.


In [17]:
set_determinism(seed=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

BASE_PATH = "/kaggle/working/brats2021"
SAVE_DIR = "/kaggle/working/brats_final_outputs"
CKPT_DIR = os.path.join(SAVE_DIR, "checkpoints")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

Device: cuda


In [18]:
BASE_PATH = "/kaggle/working/brats2021"

all_seg_files = glob.glob(BASE_PATH + "/**/*seg.nii.gz", recursive=True)
print("Segmentation files found:", len(all_seg_files))

data_dicts = []

for seg_path in all_seg_files:
    case_dir = os.path.dirname(seg_path)
    case_id = os.path.basename(seg_path).replace("_seg.nii.gz", "")

    flair = os.path.join(case_dir, case_id + "_flair.nii.gz")
    t1 = os.path.join(case_dir, case_id + "_t1.nii.gz")
    t1ce = os.path.join(case_dir, case_id + "_t1ce.nii.gz")
    t2 = os.path.join(case_dir, case_id + "_t2.nii.gz")

    if all(os.path.exists(p) for p in [flair, t1, t1ce, t2, seg_path]):
        data_dicts.append({
            "image": [flair, t1, t1ce, t2],
            "label": seg_path
        })

print("Total valid cases:", len(data_dicts))
print(data_dicts[0])

Segmentation files found: 1251
Total valid cases: 1251
{'image': ['/kaggle/working/brats2021/BraTS2021_01564/BraTS2021_01564_flair.nii.gz', '/kaggle/working/brats2021/BraTS2021_01564/BraTS2021_01564_t1.nii.gz', '/kaggle/working/brats2021/BraTS2021_01564/BraTS2021_01564_t1ce.nii.gz', '/kaggle/working/brats2021/BraTS2021_01564/BraTS2021_01564_t2.nii.gz'], 'label': '/kaggle/working/brats2021/BraTS2021_01564/BraTS2021_01564_seg.nii.gz'}


In [19]:
data_dicts = data_dicts[:500]

train_files, val_files = train_test_split(
    data_dicts,
    test_size=0.2,
    random_state=42
)

print("Training cases:", len(train_files))
print("Validation cases:", len(val_files))

Training cases: 400
Validation cases: 100


In [22]:
roi_size = (96, 96, 96)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),

    Spacingd(
        keys=["image", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=("bilinear", "nearest")
    ),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=0,
        a_max=3000,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),

    Lambdad(
        keys="label",
        func=lambda x: torch.where(x == 4, torch.tensor(3, dtype=x.dtype), x)
    ),

    CropForegroundd(keys=["image", "label"], source_key="image"),

    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=roi_size,
        pos=1,
        neg=1,
        num_samples=2,
        image_key="image",
        image_threshold=0
    ),

    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.5),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),

    EnsureTyped(keys=["image", "label"])
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),

    Spacingd(
        keys=["image", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=("bilinear", "nearest")
    ),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=0,
        a_max=3000,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),

    Lambdad(
        keys="label",
        func=lambda x: torch.where(x == 4, torch.tensor(3, dtype=x.dtype), x)
    ),

    CropForegroundd(keys=["image", "label"], source_key="image"),
    EnsureTyped(keys=["image", "label"])
])

In [23]:
train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds = Dataset(data=val_files, transform=val_transforms)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

In [24]:
sample = train_ds[0]

if isinstance(sample, list):
    sample = sample[0]

print("Image shape:", sample["image"].shape)
print("Label shape:", sample["label"].shape)
print("Unique label values:", torch.unique(sample["label"]))

Image shape: torch.Size([4, 96, 96, 96])
Label shape: torch.Size([1, 96, 96, 96])
Unique label values: metatensor([0., 1., 2., 3.])


In [25]:
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2
).to(device)

loss_function = DiceFocalLoss(
    to_onehot_y=True,
    softmax=True,
    include_background=False,
    gamma=2.0
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

dice_metric = DiceMetric(
    include_background=False,
    reduction="mean"
)

post_pred = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

In [26]:
def save_checkpoint(epoch, model, optimizer, train_loss, val_dice, best_dice, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_dice": val_dice,
        "best_dice": best_dice
    }, path)

In [ ]:
max_epochs = 100

best_dice = -1
best_epoch = -1
history = []

for epoch in range(1, max_epochs + 1):
    print(f"\nEpoch {epoch}/{max_epochs}")

    model.train()
    epoch_loss = 0

    for step, batch_data in enumerate(train_loader):
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        if step % 10 == 0:
            print(f"Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

    epoch_loss = epoch_loss / len(train_loader)

    model.eval()

    with torch.no_grad():
        for val_data in val_loader:
            val_inputs = val_data["image"].to(device)
            val_labels = val_data["label"].to(device)

            val_outputs = sliding_window_inference(
                val_inputs,
                roi_size=roi_size,
                sw_batch_size=1,
                predictor=model
            )

            val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
            val_labels = [post_label(i) for i in decollate_batch(val_labels)]

            dice_metric(y_pred=val_outputs, y=val_labels)

        val_dice = dice_metric.aggregate().item()
        dice_metric.reset()

    print(f"Average Training Loss: {epoch_loss:.4f}")
    print(f"Validation Dice Score: {val_dice:.4f}")

    history.append({
        "epoch": epoch,
        "train_loss": epoch_loss,
        "val_dice": val_dice
    })

    checkpoint_path = os.path.join(CKPT_DIR, f"checkpoint_epoch_{epoch:03d}.pth")

    save_checkpoint(
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        train_loss=epoch_loss,
        val_dice=val_dice,
        best_dice=best_dice,
        path=checkpoint_path
    )

    torch.save(
        model.state_dict(),
        os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pth")
    )

    torch.save(
        model.state_dict(),
        os.path.join(SAVE_DIR, "last_model.pth")
    )

    if val_dice > best_dice:
        best_dice = val_dice
        best_epoch = epoch

        torch.save(
            model.state_dict(),
            os.path.join(SAVE_DIR, "best_model.pth")
        )

        save_checkpoint(
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            train_loss=epoch_loss,
            val_dice=val_dice,
            best_dice=best_dice,
            path=os.path.join(SAVE_DIR, "best_checkpoint.pth")
        )

        print("Best model updated.")

    pd.DataFrame(history).to_csv(
        os.path.join(SAVE_DIR, "training_history.csv"),
        index=False
    )

print("\nTraining completed.")
print("Best Dice:", best_dice)
print("Best Epoch:", best_epoch)


Epoch 1/100
Step 0/400 | Loss: 1.3366
Step 10/400 | Loss: 1.2861
Step 20/400 | Loss: 1.2938
Step 30/400 | Loss: 1.2069
Step 40/400 | Loss: 1.1513
Step 50/400 | Loss: 1.1631
Step 60/400 | Loss: 1.2113
Step 70/400 | Loss: 1.1085
Step 80/400 | Loss: 1.1279
Step 90/400 | Loss: 1.1776
Step 100/400 | Loss: 1.1166
Step 110/400 | Loss: 1.1717
Step 120/400 | Loss: 1.0695
Step 130/400 | Loss: 1.1197
Step 140/400 | Loss: 1.0916
Step 150/400 | Loss: 1.1408
Step 160/400 | Loss: 1.0727
Step 170/400 | Loss: 1.0334
Step 180/400 | Loss: 1.1055
Step 190/400 | Loss: 1.1079
Step 200/400 | Loss: 1.0583
Step 210/400 | Loss: 1.0850
Step 220/400 | Loss: 1.1111
Step 230/400 | Loss: 0.9877
Step 240/400 | Loss: 1.0134
Step 250/400 | Loss: 1.0479
Step 260/400 | Loss: 1.0832
Step 270/400 | Loss: 1.0569
Step 280/400 | Loss: 1.0986
Step 290/400 | Loss: 1.0022
Step 300/400 | Loss: 0.9709
Step 310/400 | Loss: 0.9012
Step 320/400 | Loss: 1.0443
Step 330/400 | Loss: 1.0061
Step 340/400 | Loss: 1.0516
Step 350/400 | Los

/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out[idx_zm] += p


Average Training Loss: 1.0817
Validation Dice Score: 0.2163
Best model updated.

Epoch 2/100
Step 0/400 | Loss: 0.8715
Step 10/400 | Loss: 1.0009
Step 20/400 | Loss: 0.9596
Step 30/400 | Loss: 1.0181
Step 40/400 | Loss: 0.8477
Step 50/400 | Loss: 0.9572
Step 60/400 | Loss: 0.8841
Step 70/400 | Loss: 0.9140
Step 80/400 | Loss: 0.8199
Step 90/400 | Loss: 1.0055
Step 100/400 | Loss: 0.9094
Step 110/400 | Loss: 0.9042
Step 120/400 | Loss: 0.9083
Step 130/400 | Loss: 0.7731
Step 140/400 | Loss: 0.7885
Step 150/400 | Loss: 0.9803
Step 160/400 | Loss: 0.7472
Step 170/400 | Loss: 0.7385
Step 180/400 | Loss: 0.7280
Step 190/400 | Loss: 0.8925
Step 200/400 | Loss: 1.0328
Step 210/400 | Loss: 0.7685
Step 220/400 | Loss: 1.0311
Step 230/400 | Loss: 0.7030
Step 240/400 | Loss: 0.9374
Step 250/400 | Loss: 0.6025
Step 260/400 | Loss: 0.6053
Step 270/400 | Loss: 0.8682
Step 280/400 | Loss: 0.6360
Step 290/400 | Loss: 0.8069
Step 300/400 | Loss: 0.9431
Step 310/400 | Loss: 0.8975
Step 320/400 | Loss: 0